In [1]:
import pandas as pd
import sqlite3

data_clientes = {
    'id_cliente': [1, 2, 3, 4, 5],
    'nombre': ['Ana Lopez', 'Carlos Ruiz', 'Diana Torres', 'Eduardo Paz', 'Fernanda Lima'],
    'pais': ['Mexico', 'Colombia', 'Mexico', 'Argentina', 'Chile']
}

data_productos = {
    'id_producto': [101, 102, 103, 104],
    'categoria': ['Electronica', 'Ropa', 'Hogar', 'Deportes'],
    'precio': [1200.50, 450.00, 800.00, 600.00]
}

data_ordenes = {
    'id_orden': [1001, 1002, 1003, 1004, 1005],
    'id_cliente': [1, 2, 1, 4, 99],
    'id_producto': [101, 103, 102, 101, 104],
    'cantidad': [1, 2, 3, 1, 5]
}

df_clientes = pd.DataFrame(data_clientes)
df_productos = pd.DataFrame(data_productos)
df_ordenes = pd.DataFrame(data_ordenes)

df_clientes.to_csv('clientes.csv', index=False)
df_productos.to_csv('productos.csv', index=False)
df_ordenes.to_csv('ordenes.csv', index=False)

conn = sqlite3.connect('ecommerce.db')

df_clientes_csv = pd.read_csv('clientes.csv')
df_productos_csv = pd.read_csv('productos.csv')
df_ordenes_csv = pd.read_csv('ordenes.csv')

df_clientes_csv.to_sql('clientes', conn, if_exists='replace', index=False)
df_productos_csv.to_sql('productos', conn, if_exists='replace', index=False)
df_ordenes_csv.to_sql('ordenes', conn, if_exists='replace', index=False)

print("Base de datos 'ecommerce.db' y tablas creadas exitosamente a partir de los CSV.")

Base de datos 'ecommerce.db' y tablas creadas exitosamente a partir de los CSV.


In [2]:
query_inner = """
SELECT 
    c.nombre AS Nombre_Cliente,
    o.id_orden,
    p.categoria AS Producto_Comprado,
    o.cantidad,
    (o.cantidad * p.precio) AS Total_Pagado
FROM clientes c
INNER JOIN ordenes o ON c.id_cliente = o.id_cliente
INNER JOIN productos p ON o.id_producto = p.id_producto;
"""

df_inner = pd.read_sql_query(query_inner, conn)
display(df_inner)

,Nombre_Cliente,id_orden,Producto_Comprado,cantidad,Total_Pagado
0,Ana Lopez,1001,Electronica,1,1200.5
1,Ana Lopez,1003,Ropa,3,1350.0
2,Carlos Ruiz,1002,Hogar,2,1600.0
3,Eduardo Paz,1004,Electronica,1,1200.5


In [3]:
query_left = """
SELECT 
    c.id_cliente,
    c.nombre AS Nombre_Cliente,
    o.id_orden,
    o.cantidad
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente;
"""

df_left = pd.read_sql_query(query_left, conn)
display(df_left)

,id_cliente,Nombre_Cliente,id_orden,cantidad
0,1,Ana Lopez,1001.0,1.0
1,1,Ana Lopez,1003.0,3.0
2,2,Carlos Ruiz,1002.0,2.0
3,3,Diana Torres,NaN,NaN
4,4,Eduardo Paz,1004.0,1.0
5,5,Fernanda Lima,NaN,NaN


In [4]:
query_case = """
SELECT 
    id_producto,
    categoria,
    precio,
    CASE 
        WHEN precio < 500 THEN 'Económico'
        WHEN precio BETWEEN 500 AND 1000 THEN 'Estándar'
        ELSE 'Premium'
    END AS Rango_Precio
FROM productos;
"""

df_case = pd.read_sql_query(query_case, conn)
display(df_case)

,id_producto,categoria,precio,Rango_Precio
0,101,Electronica,1200.5,Premium
1,102,Ropa,450.0,Económico
2,103,Hogar,800.0,Estándar
3,104,Deportes,600.0,Estándar


In [5]:
query_semi_join = """
SELECT 
    id_cliente,
    nombre,
    pais
FROM clientes
WHERE id_cliente IN (
    SELECT id_cliente 
    FROM ordenes
);
"""

df_semi_join = pd.read_sql_query(query_semi_join, conn)
display(df_semi_join)

,id_cliente,nombre,pais
0,1,Ana Lopez,Mexico
1,2,Carlos Ruiz,Colombia
2,4,Eduardo Paz,Argentina


In [6]:
query_anti_join = """
SELECT 
    id_cliente,
    nombre,
    pais
FROM clientes
WHERE id_cliente NOT IN (
    SELECT id_cliente 
    FROM ordenes
);
"""

df_anti_join = pd.read_sql_query(query_anti_join, conn)
display(df_anti_join)

,id_cliente,nombre,pais
0,3,Diana Torres,Mexico
1,5,Fernanda Lima,Chile


In [7]:
query_sub_escalar = """
SELECT 
    id_producto,
    categoria,
    precio
FROM productos
WHERE precio > (
    SELECT AVG(precio) 
    FROM productos
);
"""

df_sub_escalar = pd.read_sql_query(query_sub_escalar, conn)
display(df_sub_escalar)

,id_producto,categoria,precio
0,101,Electronica,1200.5
1,103,Hogar,800.0
